# Feature Engineering: 4 Important Encoding Techniques

## Beginner-Friendly Practical Notebook

This notebook teaches four important ways to convert **categorical data into numbers**:

1. **One-Hot Encoding** — for nominal categories with a small number of categories
2. **Ordinal Encoding** — when categories have a meaningful order
3. **Binary Encoding** — useful for high-cardinality categorical features
4. **Target Encoding** — replaces categories using information from the target variable

### The big idea

Machine Learning algorithms work with numbers. But real-world datasets often contain values such as:

- Male / Female
- Red / Blue / Green
- School / Graduate / Postgraduate
- Hyderabad / Delhi / Mumbai / Chennai

Encoding converts these categorical values into numerical representations that a model can use.


## 0. Before We Start: How Do I Choose an Encoding?

Use this simple decision rule:

```text
                Categorical Feature
                         |
                 Does it have order?
                    /          \
                  YES           NO
                   |             |
               ORDINAL       How many categories?
                              /            \
                            FEW           MANY
                             |              |
                          ONE-HOT      Binary / Target*
```

`*` Target Encoding is appropriate only when a target variable is available and the encoding is performed without target leakage.

### Memory trick

**ORDER → Ordinal**  
**NO ORDER + FEW → One-Hot**  
**MANY → Binary**  
**TARGET + MANY → Target Encoding**


# 1. One-Hot Encoding

## Theory

One-Hot Encoding creates **one new column for each category**.

Suppose we have:

```text
Color
------
Red
Blue
Green
Red
```

After encoding:

```text
Color_Red  Color_Blue  Color_Green
1          0           0
0          1           0
0          0           1
1          0           0
```

There is **no artificial order** such as Red=1, Blue=2, Green=3.

### When should I use it?

Use One-Hot Encoding when:
- categories are **nominal** (no natural order)
- the number of unique categories is relatively small

Examples: Color, Gender, Department, City when there are only a few cities.

In [ ]:
import pandas as pd

# Small example dataset
df_onehot = pd.DataFrame({
    "Color": ["Red", "Blue", "Green", "Red", "Blue"]
})

print("Original data:")
display(df_onehot)

In [ ]:
# One-Hot Encoding using pandas
onehot_result = pd.get_dummies(df_onehot, columns=["Color"], dtype=int)

print("After One-Hot Encoding:")
display(onehot_result)

### What happened?

The original `Color` column disappeared and three new columns were created:

- `Color_Blue`
- `Color_Green`
- `Color_Red`

A value of **1 means the row belongs to that category**. A value of **0 means it does not**.

### Important warning

If a categorical feature has thousands of unique values, One-Hot Encoding can create thousands of columns. In that situation, consider another technique.

# 2. Ordinal Encoding

## Theory

Ordinal Encoding is used when categories have a **meaningful order**.

Example:

```text
Education
School < Graduate < Postgraduate
```

We can encode:

```text
School       → 0
Graduate     → 1
Postgraduate → 2
```

Another example:

```text
Poor < Average < Good < Excellent
```

### Key point

The numbers represent an **order**, not just labels.

### When should I use it?

Use it for features such as:
- Education level
- Satisfaction level
- Product rating
- Size: Small < Medium < Large

Do **not** use ordinary ordinal encoding for categories that have no natural order, such as colors.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

df_ordinal = pd.DataFrame({
    "Education": [
        "School",
        "Graduate",
        "Postgraduate",
        "Graduate",
        "School"
    ]
})

print("Original data:")
display(df_ordinal)

In [ ]:
# Tell the encoder the CORRECT order.
education_order = [["School", "Graduate", "Postgraduate"]]

ordinal_encoder = OrdinalEncoder(categories=education_order)

df_ordinal["Education_Encoded"] = ordinal_encoder.fit_transform(
    df_ordinal[["Education"]]
).astype(int)

print("After Ordinal Encoding:")
display(df_ordinal)

### Why did we specify the order?

Because the computer does not automatically know that:

```text
School < Graduate < Postgraduate
```

We explicitly told it the correct order.

### Common mistake

Do not do this for a nominal feature:

```text
Red   → 0
Blue  → 1
Green → 2
```

because the numbers can incorrectly suggest that Green > Blue > Red.

# 3. Binary Encoding

## Important: Binary Encoding does NOT mean only 2 categories

This is a very common confusion.

### Binary variable

A binary variable has only two categories:

```text
Male   → 1
Female → 0
```

### Binary Encoding

Binary Encoding is a technique that can encode **many categories** using binary digits.

Example:

```text
City       Number       Binary
--------------------------------
Hyderabad     1           001
Delhi         2           010
Mumbai        3           011
Chennai       4           100
```

The binary digits become separate columns.

### Why use it?

Suppose a feature has 1,000 unique cities.

- One-Hot Encoding can create around 1,000 columns.
- Binary Encoding can represent the categories using far fewer columns.

This makes Binary Encoding useful for **high-cardinality categorical features**.

In [ ]:
# Install category_encoders if it is not already installed.
# Run this cell once if needed:
# %pip install category_encoders

import category_encoders as ce

df_binary = pd.DataFrame({
    "City": [
        "Hyderabad",
        "Delhi",
        "Mumbai",
        "Chennai",
        "Pune",
        "Kolkata",
        "Bangalore",
        "Jaipur"
    ]
})

print("Original data:")
display(df_binary)

In [ ]:
binary_encoder = ce.BinaryEncoder(cols=["City"])

binary_result = binary_encoder.fit_transform(df_binary)

print("After Binary Encoding:")
display(binary_result)

### What happened internally?

Conceptually:

```text
City
 ↓
Assign an integer code
 ↓
Convert the code to binary
 ↓
Split binary digits into columns
```

### When should I use Binary Encoding?

Use it when a categorical feature has **many unique categories**, for example:

- Thousands of cities
- Product IDs
- Store IDs
- Large collections of categories

### Important distinction

**Binary Mapping:** two categories → 0/1  
**Binary Encoding:** many categories → multiple binary columns

# 4. Target Encoding

## Theory

Target Encoding replaces each category with a value calculated from the **target variable**.

Suppose we want to predict whether a customer will purchase a product:

```text
City        Purchased
---------------------
Hyderabad       1
Hyderabad       0
Hyderabad       1
Delhi           0
Delhi           0
Mumbai          1
Mumbai          1
```

For Hyderabad:

```text
(1 + 0 + 1) / 3 = 0.67
```

For Delhi:

```text
(0 + 0) / 2 = 0.00
```

For Mumbai:

```text
(1 + 1) / 2 = 1.00
```

So:

```text
Hyderabad → 0.67
Delhi     → 0.00
Mumbai    → 1.00
```

### When should I use it?

It is useful for categorical variables with many categories when the category's relationship with the target is useful.

Examples:
- City → Purchase probability
- Product → Average sales
- Teacher → Average student score

### VERY IMPORTANT: Target Leakage

Target Encoding uses the target. Therefore, we must **not use validation/test target information to create the encoding**.

A safe workflow is:

```text
Training data
     ↓
Learn target encoding
     ↓
Transform training/validation/test data
```

For production-quality modeling, use a leakage-safe implementation or cross-validation strategy.

In [ ]:
import category_encoders as ce

df_target = pd.DataFrame({
    "City": [
        "Hyderabad", "Hyderabad", "Hyderabad",
        "Delhi", "Delhi",
        "Mumbai", "Mumbai",
        "Chennai", "Chennai", "Chennai"
    ],
    "Purchased": [
        1, 0, 1,
        0, 0,
        1, 1,
        0, 1, 0
    ]
})

print("Original data:")
display(df_target)

In [ ]:
# Demonstration: calculate the simple target mean manually.
target_mean = df_target.groupby("City")["Purchased"].mean()

print("Average target value for each city:")
display(target_mean)

In [ ]:
# Target Encoding using category_encoders.
target_encoder = ce.TargetEncoder(cols=["City"])

X = df_target[["City"]]
y = df_target["Purchased"]

X_target_encoded = target_encoder.fit_transform(X, y)

print("After Target Encoding:")
display(X_target_encoded)

### Why can Target Encoding be powerful?

One-Hot Encoding only tells the model:

```text
Is this Hyderabad?
Is this Delhi?
Is this Mumbai?
```

Target Encoding can give the model information such as:

```text
Hyderabad → 0.67 purchase rate
Delhi     → 0.00 purchase rate
Mumbai    → 1.00 purchase rate
```

But remember: **the target must be handled carefully to avoid leakage and overfitting.**

# Final Comparison

| Technique | Main idea | Best situation | Example |
|---|---|---|---|
| **One-Hot** | One column per category | Few nominal categories | Color, Department |
| **Ordinal** | Ordered numbers | Categories have order | Poor, Average, Good |
| **Binary** | Categories represented using binary digits | Many categories | 1,000 cities |
| **Target** | Category represented using target information | High-cardinality + useful target relationship | City → Purchase rate |

## Easy memory trick

```text
              CATEGORICAL DATA
                      |
              Does it have ORDER?
                 /          \
               YES           NO
                |             |
             ORDINAL       FEW categories?
                            /       \
                          YES       MANY
                           |          |
                       ONE-HOT    BINARY
                                      |
                              Target available?
                                      |
                                  TARGET ENCODING
```


# Mini Practice Exercise

Try to choose the best encoding before looking at the answer.

### Question 1

`Color = Red, Blue, Green`

**Best choice:** One-Hot Encoding

Reason: Color has no natural order and there are only a few categories.

### Question 2

`Satisfaction = Poor, Average, Good, Excellent`

**Best choice:** Ordinal Encoding

Reason: The categories have a meaningful order.

### Question 3

`City = 5,000 different cities`

**Possible choice:** Binary Encoding

Reason: One-Hot Encoding could create thousands of columns.

### Question 4

`City = many cities`, target = `Customer Purchased (0/1)`

**Possible choice:** Target Encoding

Reason: The purchase behavior associated with each city can be useful, but leakage must be controlled.


# Summary

You have now learned four important encoding techniques:

### 1. One-Hot Encoding
**Nominal + few categories** → separate 0/1 columns

### 2. Ordinal Encoding
**Meaningful order** → ordered numerical values

### 3. Binary Encoding
**Many categories** → compact binary representation

### 4. Target Encoding
**Category + target relationship** → target-based numerical representation

## Final rule to remember

> **Order → Ordinal | No order + Few → One-Hot | Many → Binary | Target relationship → Target Encoding**
